# StockPilot AI
Demand forecasting and daily ordering for small food businesses.

**In Colab:** upload `stockpilot.zip` with the folder icon on the left, then run every cell from top to bottom.

In [ ]:
import os, zipfile
if os.path.exists('stockpilot.zip') and not os.path.exists('stockpilot'):
    zipfile.ZipFile('stockpilot.zip').extractall('.')
if os.path.isdir('stockpilot/stockpilot'):
    os.chdir('stockpilot')
print(os.getcwd())

## 1. Look at the data
Kaggle *Store Item Demand Forecasting Challenge*: daily sales of 50 items in 10 stores, 2013–2017.

In [ ]:
import pandas as pd
from stockpilot import config as C
raw = pd.read_csv(C.KAGGLE_TRAIN, parse_dates=['date'])
print(raw.shape)
raw.head()

In [ ]:
raw[(raw.store == 1) & (raw.item == 1)].set_index('date')['sales'].resample('W').sum().plot(title='Store 1, item 1: weekly sales', figsize=(10, 3));

## 2. Train, test on 2017 and run the digital twin
This takes about 3 minutes. It trains nine quantile models (Layer 1), tests them on 2017 against the baseline *same day last week*, and replays 2017 under the habit policy and under StockPilot (Layers 2 and 4).

In [ ]:
import train
train.main()

## 3. Results

In [ ]:
import json
res = json.load(open('artifacts/results.json'))
pd.DataFrame(res['accuracy']).T

In [ ]:
tw = res['digital_twin_2017']
rows = {'Habit +15%': tw['habit']['margin_15pct'], 'StockPilot (most profitable)': tw['stockpilot'], 'StockPilot (students first 70%)': tw['stockpilot_min_service_70pct']}
pd.DataFrame(rows).T[['units_wasted', 'stockout_day_pct', 'fill_rate_pct', 'profit_rupees']]

In [ ]:
pd.DataFrame(tw['by_product'])

## 4. Tomorrow’s order for a small shop
Uses the synthetic demo canteen log. Replace `data/canteen_log_demo_synthetic.csv` with real records (columns `date, item, sold, leftover`).

In [ ]:
from stockpilot.smallshop import tomorrow_order, next_open_day
log = pd.read_csv('data/canteen_log_demo_synthetic.csv')
items = pd.read_csv('data/canteen_items.csv')
suppliers = pd.read_csv('data/suppliers.csv')
day = next_open_day(pd.to_datetime(log.date).max().date())
print('Order for', day)
tomorrow_order(log, items, suppliers, day, min_service=0.70).drop(columns=['_qty', '_forecast'])

## 5. The dashboard
Run `streamlit run app.py` on a computer, or deploy the folder on Streamlit Community Cloud.